In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_csv('gurgaon_properties_cleaned_v1.csv')

In [4]:
df.duplicated().sum()

122

In [5]:
df.head(1)

,property_type,society,sector,price,price_per_sqft,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,floorNum,facing,agePossession,nearbyLocations,furnishDetails,features
0,flat,signature andour heights,sector 71,0.39,9846.0,396.0,Super Built up area 325(30.19 sq.m.),1.0,1.0,2,not available,1.0,NaN,0 to 1 Year Old,"['Bigbasket SPR 5K DS', 'iON Digital Zone, DPGITM', 'NH8', 'Toll Plaza', 'DPG Institute of Technology and Mgmt', 'ICICI BANK ATM', 'Park Hospital Sohna Rd', 'IGI Airport', 'HDFC Bank, Sec 59', 'Grey Orange (India), NH8', 'Tata Projects Limited Sector 71', 'NIIT, Confluence Building', 'Candor TechSpace, Sec 48', 'TATA Workshop Gurgaon - Zedex Mobility', 'IGL CNG Pump Sec 48']",[],NaN


In [6]:
# focus is on -> areaWithType, additionalRoom, agePossession, furnishDetails, features 

## areaWithType

In [7]:
df['areaWithType'].isnull().sum()

1

In [8]:
df = df.dropna(subset=['areaWithType'])

In [9]:
df.columns

Index(['property_type', 'society', 'sector', 'price', 'price_per_sqft', 'area',
       'areaWithType', 'bedRoom', 'bathroom', 'balcony', 'additionalRoom',
       'floorNum', 'facing', 'agePossession', 'nearbyLocations',
       'furnishDetails', 'features'],
      dtype='object')

In [10]:
df['areaWithType'].isnull().sum()

0

In [11]:
df.sample(5)[['price', 'area', 'areaWithType']]

,price,area,areaWithType
3419,1.40,2262.0,Super Built up area 2262(210.15 sq.m.)Built Up area: 1854 sq.ft. (172.24 sq.m.)Carpet area: 1576 sq.ft. (146.42 sq.m.)
490,2.50,2450.0,Carpet area: 2450 (227.61 sq.m.)
2815,0.85,685.0,Carpet area: 685 (63.64 sq.m.)
2622,0.80,900.0,Plot area 900(83.61 sq.m.)Built Up area: 900 sq.ft. (83.61 sq.m.)
2250,2.20,1968.0,Super Built up area 2812(261.24 sq.m.)Built Up area: 2390 sq.ft. (222.04 sq.m.)Carpet area: 1968 sq.ft. (182.83 sq.m.)


In [12]:
text = "Super Built up area 1351(125.51 sq.m.)"
match = re.search(r'Super Built up area (\d+\.?\d*)', text)
if match:
    print(float(match.group(1)))

1351.0


In [13]:
# This function extracts the Super Built up area
def get_super_built_up_area(text):
    match = re.search(r'Super Built up area (\d+\.?\d*)', text)
    if match:
        return float(match.group(1))
    return None

In [14]:
# This function extracts the Built Up area or Carpet area
def get_area(text, area_type):
    match = re.search(area_type + r'\s*:\s*(\d+\.?\d*)', text)
    if match:
        return float(match.group(1))
    return None

In [15]:
# This function checks if the area is provided in sq.m. and converts it to sqft if needed
def convert_to_sqft(text, area_value):
    if area_value is None:
        return None
    match = re.search(r'{} \((\d+\.?\d*) sq.m.\)'.format(area_value), text)
    if match:
        sq_m_value = float(match.group(1))
        return sq_m_value * 10.7639  # conversion factor from sq.m. to sqft
    return area_value

In [16]:
## This function works when you Null values in your dataset. 

# import math
# def convert_to_sqft(raw_text, area_value):
#     if not isinstance(raw_text, str):
#         return area_value
    
#     if area_value is None or (isinstance(area_value, float) and math.isnan(area_value)):
#         return None

#     pattern = rf'{int(area_value)}\s*\((\d+\.?\d*)\s*sq\.m\.\)'
#     match = re.search(pattern, raw_text)

#     if match:
#         sqm = float(match.group(1))
#         return sqm * 10.7639
    
#     return area_value

In [17]:
# Extract Super Built up area and convert to sqft if needed
df['super_built_up_area'] = df['areaWithType'].apply(get_super_built_up_area)
df['super_built_up_area_sqft'] = df.apply(
    lambda row: convert_to_sqft(row['areaWithType'], row['super_built_up_area']),
    axis=1
)


In [18]:
# Extract Built Up area and convert to sqft if needed
df['built_up_area'] = df['areaWithType'].apply(lambda x: get_area(x, 'Built Up area'))
df['built_up_area'] = df.apply(lambda x: convert_to_sqft(x['areaWithType'], x['built_up_area']), axis=1)

In [19]:
# Extract Carpet area and convert to sqft if needed
df['carpet_area'] = df['areaWithType'].apply(lambda x: get_area(x, 'Carpet area'))
df['carpet_area'] = df.apply(lambda x: convert_to_sqft(x['areaWithType'], x['carpet_area']), axis=1)

In [20]:
df['super_built_up_area'].isnull().sum()

1887

In [21]:
df['super_built_up_area']

0         325.00
1            NaN
2            NaN
3            NaN
4            NaN
5            NaN
6        3130.00
7            NaN
8            NaN
9        1545.00
10           NaN
11           NaN
12       1735.00
13           NaN
14           NaN
15           NaN
16       1970.00
17       1270.00
18           NaN
19           NaN
20       1975.00
21           NaN
22        650.00
23           NaN
24       1875.00
25       1640.00
26       1748.87
27           NaN
28       1350.00
29       2450.00
30           NaN
31           NaN
32       2191.00
33           NaN
34           NaN
35       1320.00
36           NaN
37       1380.00
38           NaN
39           NaN
40       1950.00
41           NaN
42       2225.00
43           NaN
44           NaN
45           NaN
46       1056.58
47           NaN
48           NaN
49           NaN
50        567.00
51       1711.00
52        950.00
53       1457.00
54       1507.00
55           NaN
56        525.00
57       2368.00
58           N

In [22]:
df.sample(5)[['area', 'areaWithType','super_built_up_area']]

,area,areaWithType,super_built_up_area
2612,2003.0,Super Built up area 2003(186.08 sq.m.)Built Up area: 1338.29 sq.ft. (124.33 sq.m.)Carpet area: 986.41 sq.ft. (91.64 sq.m.),2003.0
2103,1450.0,Carpet area: 1450 (134.71 sq.m.),NaN
2374,3500.0,Built Up area: 3500 (325.16 sq.m.)Carpet area: 2500 sq.ft. (232.26 sq.m.),NaN
3327,523.0,Carpet area: 523 (48.59 sq.m.),NaN
324,2018.0,Super Built up area 2018(187.48 sq.m.),2018.0


In [23]:
df[['price','property_type','area','areaWithType','super_built_up_area','built_up_area','carpet_area']].sample(5)

,price,property_type,area,areaWithType,super_built_up_area,built_up_area,carpet_area
1967,1.90,flat,2724.0,Super Built up area 2724(253.07 sq.m.),2724.0,NaN,NaN
942,2.40,flat,2132.0,Super Built up area 2132(198.07 sq.m.)Built Up area: 1800 sq.ft. (167.23 sq.m.)Carpet area: 1400 sq.ft. (130.06 sq.m.),2132.0,1800.0,1400.0
2768,4.75,house,4500.0,Built Up area: 4500 (418.06 sq.m.),NaN,4500.0,NaN
914,3.00,flat,2860.0,Carpet area: 2860 (265.7 sq.m.),NaN,NaN,2860.0
2255,0.95,flat,1450.0,Carpet area: 1450 (134.71 sq.m.),NaN,NaN,1450.0


In [24]:
df[~((df['super_built_up_area'].isnull()) | (df['built_up_area'].isnull()) | (df['carpet_area'].isnull()))][['price','property_type','area','areaWithType','super_built_up_area','built_up_area','carpet_area']].shape

(534, 7)

In [25]:
df[df['areaWithType'].str.contains('Plot')][['price','property_type','area','areaWithType','super_built_up_area','built_up_area','carpet_area']].sample(5)

,price,property_type,area,areaWithType,super_built_up_area,built_up_area,carpet_area
1482,5.5,house,20250.0,Plot area 215(179.77 sq.m.)Built Up area: 2850 sq.yards (2382.96 sq.m.)Carpet area: 2250 sq.yards (1881.29 sq.m.),NaN,2850.0,2250.0
645,8.0,house,8287.0,Plot area 362(33.63 sq.m.)Built Up area: 8286 sq.ft. (769.79 sq.m.),NaN,8286.0,NaN
594,9.0,house,3240.0,Plot area 360(301.01 sq.m.),NaN,NaN,NaN
504,1.7,house,900.0,Plot area 900(83.61 sq.m.),NaN,NaN,NaN
1576,5.6,house,3240.0,Plot area 360(301.01 sq.m.),NaN,NaN,NaN


In [26]:
df.isnull().sum()

property_type                  0
society                        1
sector                         0
price                         17
price_per_sqft                17
area                          17
areaWithType                   0
bedRoom                        0
bathroom                       0
balcony                        0
additionalRoom                 0
floorNum                      19
facing                      1105
agePossession                  1
nearbyLocations              177
furnishDetails               980
features                     635
super_built_up_area         1887
super_built_up_area_sqft    1887
built_up_area               2615
carpet_area                 1859
dtype: int64

In [27]:
df['areaWithType'].isnull().sum()

0

In [33]:
all_nan_df = df[((df['super_built_up_area'].isnull()) & (df['built_up_area'].isnull()) & (df['carpet_area'].isnull()))][['price','property_type','area','areaWithType','super_built_up_area','built_up_area','carpet_area']]

In [34]:
all_nan_df.head()

,price,property_type,area,areaWithType,super_built_up_area,built_up_area,carpet_area
4,6.00,house,2700.0,Plot area 300(250.84 sq.m.),NaN,NaN,NaN
5,4.50,house,1350.0,Plot area 150(125.42 sq.m.),NaN,NaN,NaN
11,8.90,house,2700.0,Plot area 300(250.84 sq.m.),NaN,NaN,NaN
13,0.95,house,1070.0,Plot area 1070(99.41 sq.m.),NaN,NaN,NaN
15,1.25,house,82781.0,Plot area 115(7692.86 sq.m.),NaN,NaN,NaN


In [30]:
all_nan_index = all_nan_df = df[((df['super_built_up_area'].isnull()) & (df['built_up_area'].isnull()) & (df['carpet_area'].isnull()))][['price','property_type','area','areaWithType','super_built_up_area','built_up_area','carpet_area']].index

In [31]:
# Function to extract plot area from 'areaWithType' column
def extract_plot_area(area_with_type):
    match = re.search(r'Plot area (\d+\.?\d*)', area_with_type)
    return float(match.group(1)) if match else None

In [32]:
all_nan_df['built_up_area'] = all_nan_df['areaWithType'].apply(extract_plot_area)

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices